# One unified multi-task model for all missing variables

This notebook corrects the earlier design, which trained a separate estimator for each missing column. Here, **one neural-network model** learns a shared representation of each milking record and produces five outputs:

- `cow_id` — multiclass classification
- `reproduction_status` — multiclass classification
- `lactation_number` — multiclass classification
- `days_in_milk` — regression
- `avg_milk_flow` — regression

Each output has its own head, but all heads share the same learned trunk and are optimized together. Masked losses allow partially labelled rows to contribute to every target that is present. The earlier notebooks remain useful as preprocessing and single-target baselines.


## Evaluation contract

The split is made before fitting imputers, scalers, or label maps. Only observed target values are scored. A random row split measures recovery of missing values among cows already represented in the dataset; it does **not** claim that the model can identify a completely unseen cow. Validation and test labels for cow IDs absent from training are excluded from the cow-ID metric because a closed-set classifier cannot produce an unseen class.

No target column is used as an input feature. This prevents a target from leaking into another prediction head.


In [ ]:
from pathlib import Path
import json
import random

import numpy as np
import pandas as pd
import torch
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, f1_score,
    mean_absolute_error, mean_squared_error,
)
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
elif getattr(torch.backends, "mps", None) and torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
else:
    DEVICE = torch.device("cpu")

print("PyTorch:", torch.__version__)
print("Device:", DEVICE)


In [ ]:
# Set MAX_ROWS=None for the final full-data run. The default is laptop-friendly.
MAX_ROWS = 500_000
BATCH_SIZE = 2_048
EPOCHS = 8
LEARNING_RATE = 1e-3

DATA_CANDIDATES = [
    Path("data/processed/milk_yield_cleaned_v1.csv"),
    Path("../data/processed/milk_yield_cleaned_v1.csv"),
]
DATA_PATH = next((path for path in DATA_CANDIDATES if path.exists()), None)
if DATA_PATH is None:
    raise FileNotFoundError(
        "Could not find data/processed/milk_yield_cleaned_v1.csv. "
        "Run the cleaning workflow first or update DATA_CANDIDATES."
    )

TARGETS = [
    "cow_id", "reproduction_status", "lactation_number",
    "days_in_milk", "avg_milk_flow",
]
BASE_NUMERIC_FEATURES = [
    "flow_30_60_session", "yield_first_2_min_session",
    "milk_yield", "duration_session_sec",
]
USECOLS = ["event_date", "milking", *BASE_NUMERIC_FEATURES, *TARGETS]

df = pd.read_csv(
    DATA_PATH,
    usecols=USECOLS,
    nrows=MAX_ROWS,
    dtype={"cow_id": "string", "reproduction_status": "string", "milking": "string"},
    low_memory=False,
)
df["event_date"] = pd.to_datetime(df["event_date"], errors="coerce")
for column in [*BASE_NUMERIC_FEATURES, "lactation_number", "days_in_milk", "avg_milk_flow"]:
    df[column] = pd.to_numeric(df[column], errors="coerce")

df["event_year"] = df["event_date"].dt.year
df["event_month"] = df["event_date"].dt.month
df["event_day_of_year"] = df["event_date"].dt.dayofyear
df["event_day_of_week"] = df["event_date"].dt.dayofweek
NUMERIC_FEATURES = BASE_NUMERIC_FEATURES + [
    "event_year", "event_month", "event_day_of_year", "event_day_of_week",
]

print("Rows loaded:", len(df))
display(df[TARGETS].isna().sum().rename("missing").to_frame())


In [ ]:
# Random row split is appropriate for masked-value recovery within this herd.
rng = np.random.default_rng(SEED)
indices = rng.permutation(len(df))
train_end = int(0.70 * len(indices))
validation_end = int(0.85 * len(indices))
train_idx = indices[:train_end]
validation_idx = indices[train_end:validation_end]
test_idx = indices[validation_end:]

# Fit all preprocessing on the training partition only.
train_numeric = df.iloc[train_idx][NUMERIC_FEATURES]
feature_medians = train_numeric.median().fillna(0.0)
feature_means = train_numeric.fillna(feature_medians).mean()
feature_stds = train_numeric.fillna(feature_medians).std().replace(0, 1).fillna(1.0)
milking_categories = sorted(df.iloc[train_idx]["milking"].dropna().astype(str).unique().tolist())

def build_features(frame):
    numeric = frame[NUMERIC_FEATURES].astype("float32")
    missing = numeric.isna().astype("float32").to_numpy()
    scaled = ((numeric.fillna(feature_medians) - feature_means) / feature_stds).to_numpy(dtype="float32")
    milking = frame["milking"].astype("string").fillna("__MISSING__")
    one_hot = np.column_stack([(milking == category).to_numpy(dtype="float32") for category in milking_categories])
    unknown = (~milking.isin(milking_categories)).to_numpy(dtype="float32")[:, None]
    return np.hstack([scaled, missing, one_hot, unknown]).astype("float32")

X = build_features(df)
print("Input feature matrix:", X.shape)


In [ ]:
CATEGORICAL_TARGETS = ["cow_id", "reproduction_status", "lactation_number"]
REGRESSION_TARGETS = ["days_in_milk", "avg_milk_flow"]

def sorted_training_values(series):
    values = series.iloc[train_idx].dropna().unique().tolist()
    return sorted(values)

class_values = {target: sorted_training_values(df[target]) for target in CATEGORICAL_TARGETS}
class_maps = {target: {value: index for index, value in enumerate(values)} for target, values in class_values.items()}

encoded_targets = {}
for target in CATEGORICAL_TARGETS:
    encoded_targets[target] = df[target].map(class_maps[target]).fillna(-1).astype("int64").to_numpy()
    if not class_values[target]:
        raise ValueError(f"No observed training labels for {target}")

regression_stats = {}
for target in REGRESSION_TARGETS:
    observed_train = df[target].iloc[train_idx].dropna().astype(float)
    mean = float(observed_train.mean())
    std = max(float(observed_train.std()), 1e-6)
    lower = float(observed_train.quantile(0.005))
    upper = float(observed_train.quantile(0.995))
    regression_stats[target] = {"mean": mean, "std": std, "lower": lower, "upper": upper}
    values = pd.to_numeric(df[target], errors="coerce").to_numpy(dtype="float32")
    mask = np.isfinite(values)
    standardized = np.zeros(len(df), dtype="float32")
    standardized[mask] = (values[mask] - mean) / std
    encoded_targets[target] = standardized
    encoded_targets[target + "_mask"] = mask

for target in CATEGORICAL_TARGETS:
    print(f"{target}: {len(class_values[target]):,} training classes")
print("Regression scaling:", regression_stats)


In [ ]:
def make_dataset(row_indices):
    return TensorDataset(
        torch.from_numpy(X[row_indices]),
        torch.from_numpy(encoded_targets["cow_id"][row_indices]),
        torch.from_numpy(encoded_targets["reproduction_status"][row_indices]),
        torch.from_numpy(encoded_targets["lactation_number"][row_indices]),
        torch.from_numpy(encoded_targets["days_in_milk"][row_indices]),
        torch.from_numpy(encoded_targets["days_in_milk_mask"][row_indices]),
        torch.from_numpy(encoded_targets["avg_milk_flow"][row_indices]),
        torch.from_numpy(encoded_targets["avg_milk_flow_mask"][row_indices]),
    )

train_loader = DataLoader(make_dataset(train_idx), batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
validation_loader = DataLoader(make_dataset(validation_idx), batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
test_loader = DataLoader(make_dataset(test_idx), batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

class DairyMultiTaskNet(nn.Module):
    def __init__(self, input_size, class_counts):
        super().__init__()
        self.shared = nn.Sequential(
            nn.Linear(input_size, 256), nn.ReLU(), nn.BatchNorm1d(256), nn.Dropout(0.20),
            nn.Linear(256, 128), nn.ReLU(), nn.Dropout(0.15),
        )
        self.cow_head = nn.Linear(128, class_counts["cow_id"])
        self.reproduction_head = nn.Linear(128, class_counts["reproduction_status"])
        self.lactation_head = nn.Linear(128, class_counts["lactation_number"])
        self.days_in_milk_head = nn.Linear(128, 1)
        self.avg_milk_flow_head = nn.Linear(128, 1)

    def forward(self, x):
        shared = self.shared(x)
        return {
            "cow_id": self.cow_head(shared),
            "reproduction_status": self.reproduction_head(shared),
            "lactation_number": self.lactation_head(shared),
            "days_in_milk": self.days_in_milk_head(shared).squeeze(1),
            "avg_milk_flow": self.avg_milk_flow_head(shared).squeeze(1),
        }

class_counts = {target: len(class_values[target]) for target in CATEGORICAL_TARGETS}
model = DairyMultiTaskNet(X.shape[1], class_counts).to(DEVICE)
print(model)


In [ ]:
cross_entropy = nn.CrossEntropyLoss()
huber = nn.HuberLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)
LOSS_WEIGHTS = {
    "cow_id": 1.0, "reproduction_status": 1.0, "lactation_number": 1.0,
    "days_in_milk": 1.0, "avg_milk_flow": 1.0,
}

def masked_cross_entropy(logits, labels):
    mask = labels >= 0
    return cross_entropy(logits[mask], labels[mask]) if mask.any() else logits.sum() * 0.0

def masked_huber(predictions, labels, mask):
    return huber(predictions[mask], labels[mask]) if mask.any() else predictions.sum() * 0.0

def batch_loss(outputs, batch):
    _, cow, reproduction, lactation, dim, dim_mask, flow, flow_mask = batch
    losses = {
        "cow_id": masked_cross_entropy(outputs["cow_id"], cow),
        "reproduction_status": masked_cross_entropy(outputs["reproduction_status"], reproduction),
        "lactation_number": masked_cross_entropy(outputs["lactation_number"], lactation),
        "days_in_milk": masked_huber(outputs["days_in_milk"], dim, dim_mask),
        "avg_milk_flow": masked_huber(outputs["avg_milk_flow"], flow, flow_mask),
    }
    total = sum(LOSS_WEIGHTS[name] * value for name, value in losses.items())
    return total, losses

best_validation_loss = float("inf")
best_state = None
history = []

for epoch in range(1, EPOCHS + 1):
    model.train()
    train_total = 0.0
    for raw_batch in train_loader:
        batch = tuple(tensor.to(DEVICE) for tensor in raw_batch)
        optimizer.zero_grad(set_to_none=True)
        outputs = model(batch[0])
        loss, _ = batch_loss(outputs, batch)
        loss.backward()
        optimizer.step()
        train_total += loss.item() * len(batch[0])

    model.eval()
    validation_total = 0.0
    with torch.no_grad():
        for raw_batch in validation_loader:
            batch = tuple(tensor.to(DEVICE) for tensor in raw_batch)
            loss, _ = batch_loss(model(batch[0]), batch)
            validation_total += loss.item() * len(batch[0])

    train_loss = train_total / len(train_idx)
    validation_loss = validation_total / len(validation_idx)
    history.append({"epoch": epoch, "train_loss": train_loss, "validation_loss": validation_loss})
    print(f"Epoch {epoch:02d} | train={train_loss:.4f} | validation={validation_loss:.4f}")
    if validation_loss < best_validation_loss:
        best_validation_loss = validation_loss
        best_state = {name: value.detach().cpu().clone() for name, value in model.state_dict().items()}

model.load_state_dict(best_state)
display(pd.DataFrame(history))


In [ ]:
def collect_predictions(loader):
    collected = {name: [] for name in TARGETS}
    truths = {name: [] for name in TARGETS}
    model.eval()
    with torch.no_grad():
        for raw_batch in loader:
            batch = tuple(tensor.to(DEVICE) for tensor in raw_batch)
            outputs = model(batch[0])
            for name in CATEGORICAL_TARGETS:
                collected[name].append(outputs[name].argmax(1).cpu().numpy())
            collected["days_in_milk"].append(outputs["days_in_milk"].cpu().numpy())
            collected["avg_milk_flow"].append(outputs["avg_milk_flow"].cpu().numpy())
            truths["cow_id"].append(batch[1].cpu().numpy())
            truths["reproduction_status"].append(batch[2].cpu().numpy())
            truths["lactation_number"].append(batch[3].cpu().numpy())
            truths["days_in_milk"].append(batch[4].cpu().numpy())
            truths["avg_milk_flow"].append(batch[6].cpu().numpy())
    return ({name: np.concatenate(parts) for name, parts in collected.items()},
            {name: np.concatenate(parts) for name, parts in truths.items()})

test_predictions, test_truths = collect_predictions(test_loader)
metric_rows = []
for target in CATEGORICAL_TARGETS:
    known = test_truths[target] >= 0
    metric_rows.append({
        "target": target,
        "observed_test_rows": int(known.sum()),
        "accuracy": accuracy_score(test_truths[target][known], test_predictions[target][known]),
        "balanced_accuracy": balanced_accuracy_score(test_truths[target][known], test_predictions[target][known]),
        "macro_f1": f1_score(test_truths[target][known], test_predictions[target][known], average="macro", zero_division=0),
    })

test_rows = df.iloc[test_idx]
for target in REGRESSION_TARGETS:
    known = test_rows[target].notna().to_numpy()
    stats = regression_stats[target]
    predicted = test_predictions[target][known] * stats["std"] + stats["mean"]
    actual = test_rows.loc[test_rows[target].notna(), target].to_numpy(dtype=float)
    metric_rows.append({
        "target": target,
        "observed_test_rows": int(known.sum()),
        "mae": mean_absolute_error(actual, predicted),
        "rmse": mean_squared_error(actual, predicted) ** 0.5,
    })

metrics = pd.DataFrame(metric_rows)
display(metrics)


## Interpret the results before imputing

Compare every head against a simple baseline (majority class for classification; training median for regression). Do not fill a column merely because the network produces an output. If a head does not beat its baseline or has unacceptable minority-class recall, report that limitation and leave low-confidence values missing.

The thresholds below are operating choices, not extra models. Tune them on the validation set, lock them, and report both accuracy and coverage on the untouched test set.


In [ ]:
CLASSIFICATION_THRESHOLDS = {
    "cow_id": 0.80,
    "reproduction_status": 0.55,
    "lactation_number": 0.55,
}

def predict_all_features(features):
    loader = DataLoader(TensorDataset(torch.from_numpy(features)), batch_size=BATCH_SIZE, shuffle=False)
    predictions = {name: [] for name in TARGETS}
    confidences = {name: [] for name in CATEGORICAL_TARGETS}
    model.eval()
    with torch.no_grad():
        for (x_batch,) in loader:
            outputs = model(x_batch.to(DEVICE))
            for target in CATEGORICAL_TARGETS:
                probabilities = torch.softmax(outputs[target], dim=1)
                confidence, prediction = probabilities.max(dim=1)
                predictions[target].append(prediction.cpu().numpy())
                confidences[target].append(confidence.cpu().numpy())
            for target in REGRESSION_TARGETS:
                predictions[target].append(outputs[target].cpu().numpy())
    return (
        {name: np.concatenate(parts) for name, parts in predictions.items()},
        {name: np.concatenate(parts) for name, parts in confidences.items()},
    )

all_predictions, all_confidences = predict_all_features(X)
imputed = df.copy()
imputation_summary = []

for target in CATEGORICAL_TARGETS:
    decoded = np.asarray(class_values[target], dtype=object)[all_predictions[target]]
    accepted = imputed[target].isna().to_numpy() & (all_confidences[target] >= CLASSIFICATION_THRESHOLDS[target])
    imputed.loc[accepted, target] = decoded[accepted]
    imputed[target + "_confidence"] = all_confidences[target]
    imputed[target + "_was_imputed"] = accepted
    imputation_summary.append({"target": target, "imputed": int(accepted.sum()), "remaining_missing": int(imputed[target].isna().sum())})

for target in REGRESSION_TARGETS:
    stats = regression_stats[target]
    decoded = all_predictions[target] * stats["std"] + stats["mean"]
    decoded = np.clip(decoded, stats["lower"], stats["upper"])
    if target == "days_in_milk":
        decoded = np.rint(decoded)
    accepted = imputed[target].isna().to_numpy()
    imputed.loc[accepted, target] = decoded[accepted]
    imputed[target + "_was_imputed"] = accepted
    imputation_summary.append({"target": target, "imputed": int(accepted.sum()), "remaining_missing": int(imputed[target].isna().sum())})

display(pd.DataFrame(imputation_summary))


In [ ]:
# Save the one model and its preprocessing metadata.
MODEL_DIR = Path("models") if Path("data").exists() else Path("../models")
OUTPUT_DIR = Path("data/processed") if Path("data").exists() else Path("../data/processed")
MODEL_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

checkpoint = {
    "model_state_dict": model.state_dict(),
    "input_size": X.shape[1],
    "class_values": class_values,
    "numeric_features": NUMERIC_FEATURES,
    "feature_medians": feature_medians.to_dict(),
    "feature_means": feature_means.to_dict(),
    "feature_stds": feature_stds.to_dict(),
    "milking_categories": milking_categories,
    "regression_stats": regression_stats,
    "classification_thresholds": CLASSIFICATION_THRESHOLDS,
    "seed": SEED,
}
model_path = MODEL_DIR / "dairy_multitask_model.pt"
torch.save(checkpoint, model_path)

suffix = "prototype" if MAX_ROWS is not None else "full"
output_path = OUTPUT_DIR / f"milk_yield_multitask_imputed_{suffix}.csv"
imputed.to_csv(output_path, index=False)

print("Model saved to:", model_path.resolve())
print("Imputed data saved to:", output_path.resolve())
if MAX_ROWS is not None:
    print("Prototype only: set MAX_ROWS=None and rerun before reporting final dataset totals.")


## What to report

1. This is one jointly trained multi-task neural network with a shared trunk and five heads.
2. Preprocessing is fitted on training data only, target columns are excluded from inputs, and missing labels use masked losses.
3. Report per-target metrics; there is no scientifically meaningful single accuracy that combines classifications and regressions.
4. Report confidence-threshold coverage alongside categorical accuracy.
5. State that cow-ID performance measures recovery for known cows, not recognition of unseen animals.
6. Keep earlier separate models as baselines and compare their per-target results with this unified model.
